# Corporate Signal Intelligence — Data Collection

## Objective

This notebook is responsible for the initial data collection layer of the **Corporate Signal Intelligence** project.

The goal is to collect, normalize, and validate public market and corporate data that will later be used for:

- market anomaly detection
- financial signal monitoring
- corporate risk scoring
- machine learning feature engineering
- AI-generated executive briefings with Groq
- REST API integration with the FastAPI backend

## Data Sources

The first version of the pipeline will focus on:

| Source | Purpose |
|---|---|
| Alpha Vantage | Daily market prices, volume, and company overview data |
| SEC EDGAR | Public company filings, submissions, and structured corporate disclosures |
| Neon PostgreSQL | Persistent storage for normalized market and corporate datasets |

## Initial Scope

The initial collection scope will monitor a selected group of publicly traded companies, focusing on technology and large-cap corporations.

Planned tickers:

```text
AAPL, MSFT, NVDA, GOOGL, AMZN, META, TSLA, AMD, INTC, ORCL

In [2]:
# Updated and install all the dependencies needed for the project

%pip install --upgrade pip setuptools wheel

%pip install \
    pandas \
    numpy \
    scikit-learn \
    scipy \
    joblib \
    matplotlib \
    seaborn \
    plotly \
    requests \
    dotenv

print("\nDependencies installed successfully!")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

Dependencies installed successfully!


In [3]:
# Loading the api keys from the .env file

import os
import time
from datetime import datetime, timezone

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT")

print("Alpha Vantage key loaded:", bool(ALPHA_VANTAGE_API_KEY))
print("SEC User-Agent loaded:", SEC_USER_AGENT)
print("\nReady to start the data collection process!")

python-dotenv could not parse statement starting at line 4


Alpha Vantage key loaded: True
SEC User-Agent loaded: "corporate-signal-intelligence/1.0 sidnei.almeida1806@gmail.com

Ready to start the data collection process!


In [11]:
# Testing with two companies

TICKERS_TEST = ["AAPL", "MSFT"]

SOURCE_ALPHA = "alpha_vantage"
COLLECTED_AT = datetime.now(timezone.utc).isoformat()

In [15]:
# Creating a function to collect all the data from alpha vantage

def fetch_alpha_daily(ticker: str) -> dict:
    """
    Fetch daily market data from Alpha Vantage for a single ticker.
    Uses the free TIME_SERIES_DAILY endpoint.
    """
    url = "https://www.alphavantage.co/query"
    
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker,
        "outputsize": "compact",  # últimos ~100 dias
        "apikey": ALPHA_VANTAGE_API_KEY,
    }
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    
    if "Error Message" in data:
        raise ValueError(f"Alpha Vantage error for {ticker}: {data['Error Message']}")
    
    if "Note" in data:
        raise ValueError(f"Alpha Vantage rate limit message for {ticker}: {data['Note']}")
    
    if "Information" in data:
        raise ValueError(f"Alpha Vantage information message for {ticker}: {data['Information']}")
    
    if "Time Series (Daily)" not in data:
        raise ValueError(f"Unexpected response for {ticker}: {data}")
    
    return data

In [16]:
# Function to save the json data in a  dataframe

def normalize_alpha_daily(raw_data: dict, ticker: str) -> pd.DataFrame:
    """
    Normalize Alpha Vantage daily response into a tabular DataFrame.
    """
    time_series = raw_data["Time Series (Daily)"]
    
    rows = []
    
    for date, values in time_series.items():
        rows.append({
            "ticker": ticker,
            "date": pd.to_datetime(date).date(),
            "open": float(values["1. open"]),
            "high": float(values["2. high"]),
            "low": float(values["3. low"]),
            "close": float(values["4. close"]),
            "volume": int(values["5. volume"]),
            "source": "alpha_vantage",
            "collected_at": datetime.now(timezone.utc),
        })
    
    df = pd.DataFrame(rows)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    
    return df

In [19]:
# Coolecting the test data

all_market_data = []

for ticker in TICKERS_TEST:
    print(f"Collecting Alpha Vantage data for {ticker}...")
    
    raw_data = fetch_alpha_daily(ticker)
    df_ticker = normalize_alpha_daily(raw_data, ticker)
    
    all_market_data.append(df_ticker)
    
    print(f"{ticker}: {len(df_ticker)} rows collected")
    
    time.sleep(15)

alpha_market_df = pd.concat(all_market_data, ignore_index=True)

alpha_market_df.head()
alpha_market_df.describe()

AAPL: 100 rows collected
MSFT: 100 rows collected


,open,high,low,close,volume
count,200.000000,200.000000,200.000000,200.000000,2.000000e+02
mean,341.671870,345.275857,337.941669,341.691900,4.028165e+07
std,80.052572,80.821301,79.047821,79.988472,1.587792e+07
min,247.320000,249.199900,243.420000,246.630000,8.842175e+06
25%,262.552500,266.255000,259.990000,263.180000,3.011086e+07
50%,331.067500,332.825000,327.180000,329.510000,3.793196e+07
75%,410.225000,415.042500,405.207500,411.212500,4.566810e+07
max,487.840000,489.700000,485.960000,487.710000,1.288553e+08


In [ ]:
# Saving the test data into a csv file

alpha_market_df.to_csv("data/market_data.csv", index=False)  
print("Market data saved to market_data.csv")

Market data saved to market_data.csv


In [21]:
# Let's start collecting all the tickers fot te model

def collect_alpha_market_data(
    tickers: list[str],
    outputsize: str = "full",
    sleep_seconds: int = 15,
) -> tuple[pd.DataFrame, list[dict]]:
    """
    Collect daily market data from Alpha Vantage for multiple tickers.

    Parameters
    ----------
    tickers:
        List of stock tickers to collect.
    outputsize:
        Alpha Vantage output size. Use "compact" for ~100 rows or "full" for full history.
    sleep_seconds:
        Delay between requests to reduce rate-limit risk.

    Returns
    -------
    tuple[pd.DataFrame, list[dict]]
        A normalized market DataFrame and a list of failed tickers/errors.
    """
    all_market_data = []
    failed_tickers = []

    for ticker in tickers:
        print(f"Collecting Alpha Vantage data for {ticker} | outputsize={outputsize}")

        try:
            # This reuses your existing single-ticker function logic,
            # but sends outputsize through a direct API call here.
            url = "https://www.alphavantage.co/query"

            params = {
                "function": "TIME_SERIES_DAILY",
                "symbol": ticker,
                "outputsize": outputsize,
                "apikey": ALPHA_VANTAGE_API_KEY,
            }

            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()

            raw_data = response.json()

            if "Error Message" in raw_data:
                raise ValueError(raw_data["Error Message"])

            if "Note" in raw_data:
                raise ValueError(raw_data["Note"])

            if "Information" in raw_data:
                raise ValueError(raw_data["Information"])

            if "Time Series (Daily)" not in raw_data:
                raise ValueError(f"Unexpected response: {raw_data}")

            df_ticker = normalize_alpha_daily(raw_data, ticker)

            all_market_data.append(df_ticker)

            print(
                f"  OK | {ticker}: {len(df_ticker)} rows | "
                f"{df_ticker['date'].min()} → {df_ticker['date'].max()}"
            )

        except Exception as error:
            print(f"  FAILED | {ticker}: {error}")

            failed_tickers.append({
                "ticker": ticker,
                "error": str(error),
            })

        time.sleep(sleep_seconds)

    if not all_market_data:
        return pd.DataFrame(), failed_tickers

    market_df = pd.concat(all_market_data, ignore_index=True)
    market_df = market_df.sort_values(["ticker", "date"]).reset_index(drop=True)

    return market_df, failed_tickers

## Alpha Vantage API Limitation

During the initial data collection tests, the Alpha Vantage API proved to be too limited for the historical market data requirements of this project.

Although the `TIME_SERIES_DAILY` endpoint worked for small `compact` requests, it only returned approximately the latest 100 trading days per ticker. When attempting to use `outputsize=full` to retrieve a larger historical dataset, the API returned repeated `503 Service Unavailable` errors.

Because the project requires a more reliable historical data source for anomaly detection and feature engineering, Alpha Vantage will not be used as the main market data provider in the MVP.

### Decision

For the next stage, the project will pivot to a more accessible historical market data source, while Alpha Vantage may still be used later as an optional fallback or enrichment provider.

### Impact on the Pipeline

```text
Alpha Vantage
    ↓
Limited compact data
    ↓
Unstable full historical collection
    ↓
Not suitable as the primary source

In [27]:
# Testing the stooq public endpoint

import requests

ticker = "AAPL"
symbol = f"{ticker.lower()}.us"
url = f"https://stooq.com/q/d/l/?s={symbol}&i=d"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Final URL:", response.url)
print(response.text[:1000])

Status: 200
Content-Type: text/plain; charset=UTF-8
Final URL: https://stooq.com/q/d/l/?s=aapl.us&i=d
Get your apikey:

1. Open https://stooq.com/q/d/?s=aapl.us&get_apikey
2. Enter the captcha code.
3. Copy the CSV download link at the bottom of the page - it will contain the <apikey> variable.
4. Append the <apikey> variable with its value to your requests, e.g.
   https://stooq.com/q/d/l/?s=aapl.us&i=d&apikey=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX



In [29]:
# Loading the api keys from the .env file

import os
from io import StringIO
from datetime import datetime, timezone

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

STOOQ_API_KEY = os.getenv("STOOQ_API_KEY")

print("Stooq key loaded:", bool(STOOQ_API_KEY))

python-dotenv could not parse statement starting at line 4


Stooq key loaded: True


In [30]:
def fetch_stooq_daily(ticker: str) -> pd.DataFrame:
    """
    Fetch daily historical stock data from Stooq using the CSV download API key.
    """
    if not STOOQ_API_KEY:
        raise ValueError("Missing STOQ/STOOQ API key. Check STOOQ_API_KEY in .env.")

    symbol = f"{ticker.lower()}.us"
    url = "https://stooq.com/q/d/l/"

    params = {
        "s": symbol,
        "i": "d",
        "apikey": STOOQ_API_KEY,
    }

    headers = {
        "User-Agent": "corporate-signal-intelligence/1.0"
    }

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    text = response.text.strip()

    if not text.startswith("Date,Open,High,Low,Close,Volume"):
        print("Unexpected response preview:")
        print(text[:1000])
        raise ValueError(f"Stooq did not return a valid CSV for {ticker}")

    df = pd.read_csv(StringIO(text))

    df = df.rename(
        columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )

    df["ticker"] = ticker.upper()
    df["date"] = pd.to_datetime(df["date"])
    df["source"] = "stooq"
    df["collected_at"] = datetime.now(timezone.utc)

    df = df[
        [
            "ticker",
            "date",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "source",
            "collected_at",
        ]
    ]

    return df.sort_values(["ticker", "date"]).reset_index(drop=True)

In [32]:
# Testing with AAPL ticker

aapl_stooq_df = fetch_stooq_daily("AAPL")

print(aapl_stooq_df.shape)
display(aapl_stooq_df.head())
display(aapl_stooq_df.tail())

(10507, 9)


,ticker,date,open,high,low,close,volume,source,collected_at
0,AAPL,1984-09-07,0.099172,0.100390,0.097975,0.099172,99242379,stooq,2026-05-21 19:49:53.525328+00:00
1,AAPL,1984-09-10,0.099172,0.099477,0.096788,0.098584,77028276,stooq,2026-05-21 19:49:53.525328+00:00
2,AAPL,1984-09-11,0.099477,0.102175,0.099477,0.100390,181637249,stooq,2026-05-21 19:49:53.525328+00:00
3,AAPL,1984-09-12,0.100390,0.100977,0.097367,0.097367,158675628,stooq,2026-05-21 19:49:53.525328+00:00
4,AAPL,1984-09-13,0.102784,0.103077,0.102784,0.102784,247131424,stooq,2026-05-21 19:49:53.525328+00:00


,ticker,date,open,high,low,close,volume,source,collected_at
10502,AAPL,2026-05-15,297.900,303.20,296.52,300.230,54862836,stooq,2026-05-21 19:49:53.525328+00:00
10503,AAPL,2026-05-18,300.240,300.66,294.91,297.840,34482959,stooq,2026-05-21 19:49:53.525328+00:00
10504,AAPL,2026-05-19,296.970,300.51,296.35,298.970,42243561,stooq,2026-05-21 19:49:53.525328+00:00
10505,AAPL,2026-05-20,298.180,302.80,298.08,302.250,38229843,stooq,2026-05-21 19:49:53.525328+00:00
10506,AAPL,2026-05-21,301.055,305.54,300.40,305.155,17662791,stooq,2026-05-21 19:49:53.525328+00:00


In [33]:
# Defining function to collect all the market data

TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "GOOGL",
    "AMZN",
    "META",
    "TSLA",
    "AMD",
    "INTC",
    "ORCL",
]

def collect_stooq_market_data(tickers: list[str]) -> tuple[pd.DataFrame, list[dict]]:
    all_data = []
    failed = []

    for ticker in tickers:
        print(f"Collecting Stooq data for {ticker}...")

        try:
            df_ticker = fetch_stooq_daily(ticker)
            all_data.append(df_ticker)

            print(
                f"  OK | {ticker}: {len(df_ticker)} rows | "
                f"{df_ticker['date'].min().date()} → {df_ticker['date'].max().date()}"
            )

        except Exception as error:
            print(f"  FAILED | {ticker}: {error}")
            failed.append(
                {
                    "ticker": ticker,
                    "error": str(error),
                }
            )

    if not all_data:
        return pd.DataFrame(), failed

    market_df = pd.concat(all_data, ignore_index=True)
    market_df = market_df.sort_values(["ticker", "date"]).reset_index(drop=True)

    return market_df, failed

In [34]:
stooq_market_df, stooq_failed = collect_stooq_market_data(TICKERS)

print("Final shape:", stooq_market_df.shape)
print("Failed:", stooq_failed)

display(stooq_market_df.head())
display(stooq_market_df.tail())

  OK | AAPL: 10507 rows | 1984-09-07 → 2026-05-21
  OK | MSFT: 10124 rows | 1986-03-13 → 2026-05-21
  OK | NVDA: 6874 rows | 1999-01-22 → 2026-05-21
  OK | GOOGL: 5474 rows | 2004-08-19 → 2026-05-21
  OK | AMZN: 7294 rows | 1997-05-16 → 2026-05-21
  OK | META: 3522 rows | 2012-05-18 → 2026-05-21
  OK | TSLA: 3999 rows | 2010-06-28 → 2026-05-21
  OK | AMD: 10878 rows | 1983-03-21 → 2026-05-21
  OK | INTC: 13697 rows | 1972-01-07 → 2026-05-21
  OK | ORCL: 9627 rows | 1988-03-02 → 2026-05-21
Final shape: (81996, 9)
Failed: []


,ticker,date,open,high,low,close,volume,source,collected_at
0,AAPL,1984-09-07,0.099172,0.100390,0.097975,0.099172,99242379.0,stooq,2026-05-21 19:51:02.287737+00:00
1,AAPL,1984-09-10,0.099172,0.099477,0.096788,0.098584,77028276.0,stooq,2026-05-21 19:51:02.287737+00:00
2,AAPL,1984-09-11,0.099477,0.102175,0.099477,0.100390,181637249.0,stooq,2026-05-21 19:51:02.287737+00:00
3,AAPL,1984-09-12,0.100390,0.100977,0.097367,0.097367,158675628.0,stooq,2026-05-21 19:51:02.287737+00:00
4,AAPL,1984-09-13,0.102784,0.103077,0.102784,0.102784,247131424.0,stooq,2026-05-21 19:51:02.287737+00:00


,ticker,date,open,high,low,close,volume,source,collected_at
81991,TSLA,2026-05-15,433.980,434.66,422.00,422.24,52688742.0,stooq,2026-05-21 19:51:12.058155+00:00
81992,TSLA,2026-05-18,419.270,421.13,405.33,409.99,52474188.0,stooq,2026-05-21 19:51:12.058155+00:00
81993,TSLA,2026-05-19,403.160,405.63,393.63,404.11,46500552.0,stooq,2026-05-21 19:51:12.058155+00:00
81994,TSLA,2026-05-20,407.600,417.46,406.39,417.26,45294745.0,stooq,2026-05-21 19:51:12.058155+00:00
81995,TSLA,2026-05-21,422.175,426.95,412.90,419.65,29472948.0,stooq,2026-05-21 19:51:12.058155+00:00


In [35]:
# Verifying the converage of the tickers in the dataframe

stooq_market_df.groupby("ticker").agg(
    rows=("date", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
)

,rows,min_date,max_date
ticker,,,
AAPL,10507,1984-09-07,2026-05-21
AMD,10878,1983-03-21,2026-05-21
AMZN,7294,1997-05-16,2026-05-21
GOOGL,5474,2004-08-19,2026-05-21
INTC,13697,1972-01-07,2026-05-21
META,3522,2012-05-18,2026-05-21
MSFT,10124,1986-03-13,2026-05-21
NVDA,6874,1999-01-22,2026-05-21
ORCL,9627,1988-03-02,2026-05-21


In [37]:
stooq_market_df.to_csv("data/stooq_market_raw.csv", index=False)

## Market Data Source Update — Stooq API

After testing Alpha Vantage as the initial market data provider, the API proved to be too limited for the historical data requirements of this project.

The `compact` mode returned only a small recent sample, while attempts to collect full historical data resulted in repeated service errors and quota-related limitations. Because the anomaly detection pipeline requires a broader historical base, Alpha Vantage is not suitable as the main market data source for this project.

### New Data Source: Stooq

The data collection pipeline was updated to use the **Stooq CSV API** with an API key.

This approach worked successfully and returned a much larger historical dataset for the selected companies, including several decades of daily market data for many tickers.

### Result

The new Stooq-based collection method successfully retrieved historical daily market data for the following companies:

```text
AAPL, MSFT, NVDA, GOOGL, AMZN, META, TSLA, AMD, INTC, ORCL

In [5]:
# Defining the SEC User-Agent the next day

import os
import requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

SEC_USER_AGENT = os.getenv("SEC_USER_AGENT")

print("SEC User-Agent loaded:", SEC_USER_AGENT)

python-dotenv could not parse statement starting at line 4


SEC User-Agent loaded: "corporate-signal-intelligence/1.0 sidnei.almeida1806@gmail.com


In [6]:
# Defining the base for the SEC EDGAR public endpoints

def sec_get_json(url: str) -> dict:
    """
    Make a GET request to SEC EDGAR public endpoints.
    """
    if not SEC_USER_AGENT:
        raise ValueError("Missing SEC_USER_AGENT in .env")

    headers = {
        "User-Agent": SEC_USER_AGENT,
        "Accept-Encoding": "gzip, deflate",
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    return response.json()

In [10]:
# Defining the function to look for the CIK of the tickers

def fetch_sec_company_tickers() -> pd.DataFrame:
    """
    Fetch SEC ticker to CIK mapping.
    """
    url = "https://www.sec.gov/files/company_tickers.json"

    data = sec_get_json(url)

    rows = []

    for _, item in data.items():
        rows.append({
            "ticker": item["ticker"],
            "cik": str(item["cik_str"]).zfill(10),
            "company_name": item["title"],
            "source": "sec_edgar",
            "collected_at": datetime.now(timezone.utc),
        })

    df = pd.DataFrame(rows)

    return df.sort_values("ticker").reset_index(drop=True)

In [11]:
# Starting to test the collection with AAPL

sec_companies_df = fetch_sec_company_tickers()

aapl_company = sec_companies_df[sec_companies_df["ticker"] == "AAPL"]

aapl_company

,ticker,cik,company_name,source,collected_at
25,AAPL,0000320193,Apple Inc.,sec_edgar,2026-05-21 19:59:24.334377+00:00


In [12]:
# Defining the functiom to collec submissions of the AAPL with the CIK of the company

def fetch_sec_submissions(cik: str) -> dict:
    """
    Fetch SEC company submissions by CIK.
    """
    cik = str(cik).zfill(10)

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    return sec_get_json(url)

In [13]:
aapl_cik = aapl_company.iloc[0]["cik"]

aapl_submissions_raw = fetch_sec_submissions(aapl_cik)

aapl_submissions_raw.keys()

dict_keys(['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings'])

In [14]:
# Defining the fuction to normalize SEC submissions into a tabular DataFrame

def normalize_sec_recent_filings(submissions_raw: dict, ticker: str, cik: str) -> pd.DataFrame:
    """
    Normalize recent SEC filings from company submissions.
    """
    recent = submissions_raw["filings"]["recent"]

    rows = []

    total_filings = len(recent["accessionNumber"])

    for i in range(total_filings):
        accession_number = recent["accessionNumber"][i]
        primary_document = recent["primaryDocument"][i]

        accession_clean = accession_number.replace("-", "")

        filing_url = (
            f"https://www.sec.gov/Archives/edgar/data/"
            f"{int(cik)}/{accession_clean}/{primary_document}"
        )

        rows.append({
            "ticker": ticker,
            "cik": cik,
            "accession_number": accession_number,
            "filing_date": recent["filingDate"][i],
            "report_date": recent["reportDate"][i],
            "form_type": recent["form"][i],
            "primary_document": primary_document,
            "filing_url": filing_url,
            "source": "sec_edgar",
            "collected_at": datetime.now(timezone.utc),
        })

    df = pd.DataFrame(rows)

    df["filing_date"] = pd.to_datetime(df["filing_date"])
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")

    return df.sort_values("filing_date", ascending=False).reset_index(drop=True)

In [15]:
aapl_filings_df = normalize_sec_recent_filings(
    submissions_raw=aapl_submissions_raw,
    ticker="AAPL",
    cik=aapl_cik,
)

aapl_filings_df.head(20)

,ticker,cik,accession_number,filing_date,report_date,form_type,primary_document,filing_url,source,collected_at
0,AAPL,0000320193,0001140361-26-020871,2026-05-12,2026-05-08,4,xslF345X06/form4.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507929+00:00
1,AAPL,0000320193,0001140361-26-020298,2026-05-08,2026-05-06,4,xslF345X06/form4.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507934+00:00
2,AAPL,0000320193,0001921094-26-000446,2026-05-06,NaT,144,xsl144X01/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507937+00:00
3,AAPL,0000320193,0001950047-26-004044,2026-05-05,NaT,144,xsl144X01/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507938+00:00
4,AAPL,0000320193,0000320193-26-000013,2026-05-01,2026-03-28,10-Q,aapl-20260328.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507940+00:00
5,AAPL,0000320193,0000320193-26-000011,2026-04-30,2026-04-30,8-K,aapl-20260430.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507941+00:00
6,AAPL,0000320193,0002100119-26-000139,2026-04-29,NaT,SCHEDULE 13G,xslSCHEDULE_13G_X02/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507943+00:00
7,AAPL,0000320193,0001140361-26-017175,2026-04-27,2026-04-23,4,xslF345X06/form4.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507944+00:00
8,AAPL,0000320193,0001950047-26-003721,2026-04-23,NaT,144,xsl144X01/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507946+00:00
9,AAPL,0000320193,0001140361-26-015711,2026-04-20,2026-04-17,8-K,ef20071035_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:01:10.507947+00:00


In [16]:
# Summary

aapl_filings_df.groupby("form_type").agg(
    rows=("accession_number", "count"),
    latest_filing=("filing_date", "max"),
).sort_values("rows", ascending=False).head(20)

,rows,latest_filing
form_type,,
4,586,2026-05-12
8-K,105,2026-04-30
424B2,50,2025-05-06
144,43,2026-05-06
10-Q,33,2026-05-01
PX14A6G,27,2026-01-22
FWP,25,2025-05-06
SC 13G/A,22,2024-02-14
DEFA14A,12,2026-01-08


In [17]:
# Collecting facts/XBRL data from AAPL

def fetch_sec_company_facts(cik: str) -> dict:
    """
    Fetch SEC XBRL company facts by CIK.
    """
    cik = str(cik).zfill(10)

    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    return sec_get_json(url)

In [18]:
# Starting the data collection

aapl_company_facts_raw = fetch_sec_company_facts(aapl_cik)

aapl_company_facts_raw.keys()

dict_keys(['cik', 'entityName', 'facts'])

In [19]:
# Setting the target concepts for fact collections just for AAPL

TARGET_CONCEPTS = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity",
    "CashAndCashEquivalentsAtCarryingValue",
    "OperatingIncomeLoss",
    "ResearchAndDevelopmentExpense",
]

In [20]:
# Normalizing the facts into a tabular DataFrame

def normalize_sec_company_facts(
    company_facts_raw: dict,
    ticker: str,
    cik: str,
    target_concepts: list[str],
) -> pd.DataFrame:
    """
    Normalize selected SEC company facts from XBRL data.
    """
    rows = []

    facts = company_facts_raw.get("facts", {})
    us_gaap = facts.get("us-gaap", {})

    for concept in target_concepts:
        if concept not in us_gaap:
            continue

        concept_data = us_gaap[concept]
        label = concept_data.get("label")
        description = concept_data.get("description")
        units = concept_data.get("units", {})

        for unit, observations in units.items():
            for obs in observations:
                rows.append({
                    "ticker": ticker,
                    "cik": cik,
                    "taxonomy": "us-gaap",
                    "concept": concept,
                    "label": label,
                    "description": description,
                    "unit": unit,
                    "value": obs.get("val"),
                    "start_date": obs.get("start"),
                    "end_date": obs.get("end"),
                    "filing_date": obs.get("filed"),
                    "form_type": obs.get("form"),
                    "fiscal_year": obs.get("fy"),
                    "fiscal_period": obs.get("fp"),
                    "source": "sec_edgar",
                    "collected_at": datetime.now(timezone.utc),
                })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
        df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")
        df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce")

        df = df.sort_values(
            ["concept", "end_date", "filing_date"],
            ascending=[True, False, False],
        ).reset_index(drop=True)

    return df

In [22]:
# Starting the collection

aapl_facts_df = normalize_sec_company_facts(
    company_facts_raw=aapl_company_facts_raw,
    ticker="AAPL",
    cik=aapl_cik,
    target_concepts=TARGET_CONCEPTS,
)

print(aapl_facts_df.shape)

aapl_facts_df.head(20)

(1688, 16)


,ticker,cik,taxonomy,concept,label,description,unit,value,start_date,end_date,filing_date,form_type,fiscal_year,fiscal_period,source,collected_at
0,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,371082000000,NaT,2026-03-28,2026-05-01,10-Q,2026.0,Q2,sec_edgar,2026-05-21 20:04:00.665758+00:00
1,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,379297000000,NaT,2025-12-27,2026-01-30,10-Q,2026.0,Q1,sec_edgar,2026-05-21 20:04:00.665757+00:00
2,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,359241000000,NaT,2025-09-27,2026-05-01,10-Q,2026.0,Q2,sec_edgar,2026-05-21 20:04:00.665756+00:00
3,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,359241000000,NaT,2025-09-27,2026-01-30,10-Q,2026.0,Q1,sec_edgar,2026-05-21 20:04:00.665755+00:00
4,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,359241000000,NaT,2025-09-27,2025-10-31,10-K,2025.0,FY,sec_edgar,2026-05-21 20:04:00.665754+00:00
5,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,331495000000,NaT,2025-06-28,2025-08-01,10-Q,2025.0,Q3,sec_edgar,2026-05-21 20:04:00.665753+00:00
6,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,331233000000,NaT,2025-03-29,2025-05-02,10-Q,2025.0,Q2,sec_edgar,2026-05-21 20:04:00.665752+00:00
7,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,344085000000,NaT,2024-12-28,2025-01-31,10-Q,2025.0,Q1,sec_edgar,2026-05-21 20:04:00.665750+00:00
8,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,364980000000,NaT,2024-09-28,2025-10-31,10-K,2025.0,FY,sec_edgar,2026-05-21 20:04:00.665749+00:00
9,AAPL,0000320193,us-gaap,Assets,Assets,Sum of the carrying amounts as of the balance ...,USD,364980000000,NaT,2024-09-28,2025-08-01,10-Q,2025.0,Q3,sec_edgar,2026-05-21 20:04:00.665748+00:00


In [23]:
aapl_facts_df.groupby(["concept", "unit"]).agg(
    rows=("value", "count"),
    min_end_date=("end_date", "min"),
    max_end_date=("end_date", "max"),
    latest_filing=("filing_date", "max"),
).sort_values("rows", ascending=False)

,,rows,min_end_date,max_end_date,latest_filing
concept,unit,,,,
NetIncomeLoss,USD,334,2007-09-29,2026-03-28,2026-05-01
StockholdersEquity,USD,258,2006-09-30,2026-03-28,2026-05-01
ResearchAndDevelopmentExpense,USD,230,2007-09-29,2026-03-28,2026-05-01
OperatingIncomeLoss,USD,230,2007-09-29,2026-03-28,2026-05-01
CashAndCashEquivalentsAtCarryingValue,USD,226,2006-09-30,2026-03-28,2026-05-01
Assets,USD,144,2008-09-27,2026-03-28,2026-05-01
Liabilities,USD,142,2008-09-27,2026-03-28,2026-05-01
RevenueFromContractWithCustomerExcludingAssessedTax,USD,113,2017-09-30,2026-03-28,2026-05-01
Revenues,USD,11,2016-09-24,2018-09-29,2018-11-05


In [24]:
# Testing the SEC submission/fillings endpoint with the CIK

aapl_cik = aapl_company.iloc[0]["cik"]

aapl_submissions_raw = fetch_sec_submissions(aapl_cik)

print(aapl_submissions_raw.keys())
print("Company name:", aapl_submissions_raw.get("name"))
print("CIK:", aapl_submissions_raw.get("cik"))
print("Tickers:", aapl_submissions_raw.get("tickers"))
print("Exchanges:", aapl_submissions_raw.get("exchanges"))

dict_keys(['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings'])
Company name: Apple Inc.
CIK: 0000320193
Tickers: ['AAPL']
Exchanges: ['Nasdaq']


In [25]:
# Normalizing the fillings

aapl_filings_df = normalize_sec_recent_filings(
    submissions_raw=aapl_submissions_raw,
    ticker="AAPL",
    cik=aapl_cik,
)

print(aapl_filings_df.shape)

aapl_filings_df.head(20)

(1000, 10)


,ticker,cik,accession_number,filing_date,report_date,form_type,primary_document,filing_url,source,collected_at
0,AAPL,0000320193,0001140361-26-020871,2026-05-12,2026-05-08,4,xslF345X06/form4.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159816+00:00
1,AAPL,0000320193,0001140361-26-020298,2026-05-08,2026-05-06,4,xslF345X06/form4.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159820+00:00
2,AAPL,0000320193,0001921094-26-000446,2026-05-06,NaT,144,xsl144X01/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159822+00:00
3,AAPL,0000320193,0001950047-26-004044,2026-05-05,NaT,144,xsl144X01/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159823+00:00
4,AAPL,0000320193,0000320193-26-000013,2026-05-01,2026-03-28,10-Q,aapl-20260328.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159824+00:00
5,AAPL,0000320193,0000320193-26-000011,2026-04-30,2026-04-30,8-K,aapl-20260430.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159826+00:00
6,AAPL,0000320193,0002100119-26-000139,2026-04-29,NaT,SCHEDULE 13G,xslSCHEDULE_13G_X02/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159827+00:00
7,AAPL,0000320193,0001140361-26-017175,2026-04-27,2026-04-23,4,xslF345X06/form4.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159829+00:00
8,AAPL,0000320193,0001950047-26-003721,2026-04-23,NaT,144,xsl144X01/primary_doc.xml,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159830+00:00
9,AAPL,0000320193,0001140361-26-015711,2026-04-20,2026-04-17,8-K,ef20071035_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:06:11.159831+00:00


In [26]:
# Checking the forms

aapl_filings_df.groupby("form_type").agg(
    rows=("accession_number", "count"),
    latest_filing=("filing_date", "max"),
).sort_values("rows", ascending=False).head(20)

,rows,latest_filing
form_type,,
4,586,2026-05-12
8-K,105,2026-04-30
424B2,50,2025-05-06
144,43,2026-05-06
10-Q,33,2026-05-01
PX14A6G,27,2026-01-22
FWP,25,2025-05-06
SC 13G/A,22,2024-02-14
DEFA14A,12,2026-01-08


## SEC EDGAR Collection Test — Initial Insights

The first SEC EDGAR collection test was executed using **Apple Inc. (AAPL)** as the validation ticker.

The goal of this test was to confirm whether the SEC public endpoints could provide enough corporate data to support the **Corporate Signal Intelligence** pipeline.

### 1. Ticker to CIK Mapping

The SEC company ticker mapping worked successfully.

For Apple, the pipeline correctly identified:

```text
Ticker: AAPL
CIK: 0000320193
Company Name: Apple Inc.
Source: SEC EDGAR

In [27]:
# Defiining the tickers for all the companies

TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "GOOGL",
    "AMZN",
    "META",
    "TSLA",
    "AMD",
    "INTC",
    "ORCL",
]

In [30]:
# Configuring the SEC EDGAR Batch Collection of all 10 Companies

TARGET_CONCEPTS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "NetIncomeLoss",
    "OperatingIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity",
    "CashAndCashEquivalentsAtCarryingValue",
    "ResearchAndDevelopmentExpense",
]

IMPORTANT_FORMS = ["10-K", "10-Q", "8-K"]

sec_companies_selected_df = sec_companies_df[
    sec_companies_df["ticker"].isin(TICKERS)
].copy()

sec_companies_selected_df = sec_companies_selected_df.sort_values("ticker").reset_index(drop=True)

sec_companies_selected_df

,ticker,cik,company_name,source,collected_at
0,AAPL,0000320193,Apple Inc.,sec_edgar,2026-05-21 19:59:24.334377+00:00
1,AMD,0000002488,ADVANCED MICRO DEVICES INC,sec_edgar,2026-05-21 19:59:24.334385+00:00
2,AMZN,0001018724,AMAZON COM INC,sec_edgar,2026-05-21 19:59:24.334378+00:00
3,GOOGL,0001652044,Alphabet Inc.,sec_edgar,2026-05-21 19:59:24.334375+00:00
4,INTC,0000050863,INTEL CORP,sec_edgar,2026-05-21 19:59:24.334388+00:00
5,META,0001326801,"Meta Platforms, Inc.",sec_edgar,2026-05-21 19:59:24.334380+00:00
6,MSFT,0000789019,MICROSOFT CORP,sec_edgar,2026-05-21 19:59:24.334378+00:00
7,NVDA,0001045810,NVIDIA CORP,sec_edgar,2026-05-21 19:59:24.334370+00:00
8,ORCL,0001341439,ORACLE CORP,sec_edgar,2026-05-21 19:59:24.334389+00:00
9,TSLA,0001318605,"Tesla, Inc.",sec_edgar,2026-05-21 19:59:24.334381+00:00


In [31]:
# Collecting fillings and the companies facts

import time

all_sec_filings = []
all_sec_facts = []
sec_failed = []

for _, company in sec_companies_selected_df.iterrows():
    ticker = company["ticker"]
    cik = company["cik"]

    print(f"Collecting SEC data for {ticker} | CIK={cik}")

    try:
        # 1. Submissions / filings
        submissions_raw = fetch_sec_submissions(cik)

        filings_df = normalize_sec_recent_filings(
            submissions_raw=submissions_raw,
            ticker=ticker,
            cik=cik,
        )

        all_sec_filings.append(filings_df)

        print(f"  Filings OK: {len(filings_df)} rows")

        # 2. Company facts / XBRL
        company_facts_raw = fetch_sec_company_facts(cik)

        facts_df = normalize_sec_company_facts(
            company_facts_raw=company_facts_raw,
            ticker=ticker,
            cik=cik,
            target_concepts=TARGET_CONCEPTS,
        )

        all_sec_facts.append(facts_df)

        print(f"  Facts OK: {len(facts_df)} rows")

    except Exception as error:
        print(f"  FAILED | {ticker}: {error}")

        sec_failed.append({
            "ticker": ticker,
            "cik": cik,
            "error": str(error),
        })

    # SEC-friendly delay
    time.sleep(1)

sec_filings_df = pd.concat(all_sec_filings, ignore_index=True) if all_sec_filings else pd.DataFrame()
sec_facts_df = pd.concat(all_sec_facts, ignore_index=True) if all_sec_facts else pd.DataFrame()

print("SEC filings shape:", sec_filings_df.shape)
print("SEC facts shape:", sec_facts_df.shape)
print("Failed:", sec_failed)

  Filings OK: 1000 rows
  Facts OK: 1688 rows
  Filings OK: 1000 rows
  Facts OK: 1455 rows
  Filings OK: 1003 rows
  Facts OK: 1482 rows
  Filings OK: 1000 rows
  Facts OK: 1065 rows
  Filings OK: 1009 rows
  Facts OK: 1322 rows
  Filings OK: 1003 rows
  Facts OK: 1375 rows
  Filings OK: 1002 rows
  Facts OK: 1747 rows
  Filings OK: 1002 rows
  Facts OK: 1735 rows
  Filings OK: 1002 rows
  Facts OK: 1314 rows
  Filings OK: 1001 rows
  Facts OK: 1614 rows
SEC filings shape: (10022, 10)
SEC facts shape: (14797, 16)
Failed: []


In [32]:
# Filtering the fillings by form type

sec_important_filings_df = sec_filings_df[
    sec_filings_df["form_type"].isin(IMPORTANT_FORMS)
].copy()

sec_important_filings_df = sec_important_filings_df.sort_values(
    ["ticker", "filing_date"],
    ascending=[True, False]
).reset_index(drop=True)

sec_important_filings_df.head(20)

,ticker,cik,accession_number,filing_date,report_date,form_type,primary_document,filing_url,source,collected_at
0,AAPL,0000320193,0000320193-26-000013,2026-05-01,2026-03-28,10-Q,aapl-20260328.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868771+00:00
1,AAPL,0000320193,0000320193-26-000011,2026-04-30,2026-04-30,8-K,aapl-20260430.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868772+00:00
2,AAPL,0000320193,0001140361-26-015711,2026-04-20,2026-04-17,8-K,ef20071035_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868776+00:00
3,AAPL,0000320193,0001140361-26-006577,2026-02-24,2026-02-24,8-K,ef20060722_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868795+00:00
4,AAPL,0000320193,0000320193-26-000006,2026-01-30,2025-12-27,10-Q,aapl-20251227.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868804+00:00
5,AAPL,0000320193,0000320193-26-000005,2026-01-29,2026-01-29,8-K,aapl-20260129.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868805+00:00
6,AAPL,0000320193,0001140361-26-000199,2026-01-02,2025-12-30,8-K,ef20060722_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868811+00:00
7,AAPL,0000320193,0001140361-25-044561,2025-12-05,2025-12-04,8-K,ef20060722_8k.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868812+00:00
8,AAPL,0000320193,0000320193-25-000079,2025-10-31,2025-09-27,10-K,aapl-20250927.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868816+00:00
9,AAPL,0000320193,0000320193-25-000077,2025-10-30,2025-10-30,8-K,aapl-20251030.htm,https://www.sec.gov/Archives/edgar/data/320193...,sec_edgar,2026-05-21 20:10:02.868817+00:00


In [33]:
# Validation summary

sec_filings_summary = sec_filings_df.groupby(["ticker", "form_type"]).agg(
    rows=("accession_number", "count"),
    latest_filing=("filing_date", "max"),
).reset_index().sort_values(["ticker", "rows"], ascending=[True, False])

sec_filings_summary.head(50)

,ticker,form_type,rows,latest_filing
8,AAPL,4,586,2026-05-12
13,AAPL,8-K,105,2026-04-30
10,AAPL,424B2,50,2025-05-06
2,AAPL,144,43,2026-05-06
1,AAPL,10-Q,33,2026-05-01
23,AAPL,PX14A6G,27,2026-01-22
20,AAPL,FWP,25,2025-05-06
29,AAPL,SC 13G/A,22,2024-02-14
19,AAPL,DEFA14A,12,2026-01-08
0,AAPL,10-K,11,2025-10-31


In [34]:
sec_companies_selected_df.to_csv("data/sec_companies_selected.csv", index=False)
sec_filings_df.to_csv("data/sec_filings_raw.csv", index=False)
sec_important_filings_df.to_csv("data/sec_important_filings.csv", index=False)
sec_facts_df.to_csv("data/sec_company_facts_raw.csv", index=False)

print("SEC datasets saved.")

SEC datasets saved.


## SEC EDGAR Batch Collection — Final Notes

The SEC EDGAR collection stage was successfully validated and expanded from a single-company test using AAPL to the full monitored company list.

The pipeline now supports the collection of corporate disclosure data for the selected tickers:

```text
AAPL, MSFT, NVDA, GOOGL, AMZN, META, TSLA, AMD, INTC, ORCL